In [10]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.resolve()
sys.path.insert(0, str(project_root))

# Avoid collisions with similarly named third-party modules in notebook contexts
if "data" in sys.modules:
    del sys.modules["data"]

from data.load import load_dataset, load_config
from data.preprocess import preprocess_train, preprocess_test
from features.health_index import build_health_index
from features.velocity import build_velocity
from features.variability import build_variability
from model.clustering import build_clustering
from model.risk import build_risk_score
from model.rul import build_rul_model

config = load_config(project_root / "config" / "config.yaml")
config["dataset"]["raw_path"] = str(project_root / "data" / "raw")

train_raw, test_raw, rul_raw = load_dataset(config)
train_proc, scaler, sensor_cols = preprocess_train(train_raw, config)
test_proc = preprocess_test(test_raw, config, scaler)

# Reconstruct cycle-level test RUL using provided end-of-trajectory offsets
test_max_cycle = test_proc.groupby("unit")["cycle"].max().rename("max_cycle")
test_proc = test_proc.merge(test_max_cycle, on="unit", how="left")
test_proc["RUL"] = (test_proc["max_cycle"] - test_proc["cycle"]) + test_proc["unit"].map(rul_raw)
test_proc = test_proc.drop(columns=["max_cycle"])

train_hi, test_hi, hi_art = build_health_index(train_proc, test_proc, config)
train_vel, test_vel, vel_art = build_velocity(train_hi, test_hi, config)
train_var, test_var, var_art = build_variability(train_vel, test_vel, config)
train_cl, test_cl, cl_art = build_clustering(train_var, test_var, config)
train_rs, test_rs, risk_art = build_risk_score(train_cl, test_cl, cl_art)

preds_df, rul_art = build_rul_model(train_rs, test_rs, config)

# Shape Checks
assert len(preds_df) == 100  # one prediction per test engine
assert set(preds_df.columns) == {"unit", "true_RUL", "predicted_RUL", "error", "model_used"}

# Artifact Checks
assert rul_art.best_model is not None
assert rul_art.best_model_name in ["linear_regression", "random_forest", "gradient_boosting"]
assert len(rul_art.evaluation_metrics) == 3
assert len(rul_art.feature_importance) == 2  # RF and GB only

# Predictions Sanity
assert (preds_df["predicted_RUL"] >= 0).all()  # RUL cannot be negative

print("All checks passed successfully!")
print(f"\nBest Model: {rul_art.best_model_name}")
print(f"Model Used: {rul_art.model_used}")
print("Evaluation Metrics:")
for name, metrics in rul_art.evaluation_metrics.items():
    print(f" {name:30s} RMSE: {metrics['rmse']:.2f} NASA: {metrics['nasa_score']:.1f}")

print(f"\nFeature Importance ({rul_art.best_model_name}):")
if rul_art.best_model_name in rul_art.feature_importance:
    print(rul_art.feature_importance[rul_art.best_model_name])

print("\nSample Predictions (top 5 by risk):")
print(preds_df.sort_values("true_RUL").head(5).to_string(index=False))

[load] Train shape : (20631, 26)
[load] Test shape  : (13096, 26)
[load] RUL entries : 100
All checks passed successfully!

Best Model: gradient_boosting
Model Used: gradient_boosting
Evaluation Metrics:
 linear_regression              RMSE: 19.49 NASA: 608.2
 random_forest                  RMSE: 19.39 NASA: 832.0
 gradient_boosting              RMSE: 18.60 NASA: 698.0

Feature Importance (gradient_boosting):
risk_score        0.681477
health_index      0.203469
HI_velocity       0.111475
HI_variability    0.003578
Name: gradient_boosting_importance, dtype: float64

Sample Predictions (top 5 by risk):
 unit  true_RUL  predicted_RUL  error        model_used
   34         7              4  -3.21 gradient_boosting
   68         8             11   2.96 gradient_boosting
   31         8             13   5.16 gradient_boosting
   81         8              7  -1.49 gradient_boosting
   82         9              6  -2.89 gradient_boosting


In [9]:
import sys
from pathlib import Path

project_root = Path.cwd().parent.resolve()
sys.path.insert(0, str(project_root))

# Avoid collisions with similarly named third-party modules in notebook contexts
if "data" in sys.modules:
    del sys.modules["data"]

from data.load import load_dataset, load_config
from data.preprocess import preprocess_train, preprocess_test
from features.health_index import build_health_index
from features.velocity import build_velocity
from features.variability import build_variability
from model.clustering import build_clustering
from model.risk import build_risk_score
from model.rul import build_rul_model

config = load_config(project_root / "config" / "config.yaml")
config["dataset"]["raw_path"] = str(project_root / "data" / "raw")

train_raw, test_raw, rul_raw = load_dataset(config)
train_proc, scaler, sensor_cols = preprocess_train(train_raw, config)
test_proc = preprocess_test(test_raw, config, scaler)

# Reconstruct cycle-level test RUL using provided end-of-trajectory offsets
test_max_cycle = test_proc.groupby("unit")["cycle"].max().rename("max_cycle")
test_proc = test_proc.merge(test_max_cycle, on="unit", how="left")
test_proc["RUL"] = (test_proc["max_cycle"] - test_proc["cycle"]) + test_proc["unit"].map(rul_raw)
test_proc = test_proc.drop(columns=["max_cycle"])

train_hi, test_hi, hi_art = build_health_index(train_proc, test_proc, config)
train_vel, test_vel, vel_art = build_velocity(train_hi, test_hi, config)
train_var, test_var, var_art = build_variability(train_vel, test_vel, config)
train_cl, test_cl, cl_art = build_clustering(train_var, test_var, config)
train_rs, test_rs, risk_art = build_risk_score(train_cl, test_cl, cl_art)

preds_df, rul_art = build_rul_model(train_rs, test_rs, config)

# Shape Checks
assert len(preds_df) == 100  # one prediction per test engine
assert set(preds_df.columns) == {"unit", "true_RUL", "predicted_RUL", "error", "model_used"}

# Artifact Checks
assert rul_art.best_model is not None
assert rul_art.best_model_name in ["linear_regression", "random_forest", "gradient_boosting"]
assert len(rul_art.evaluation_metrics) == 3
assert len(rul_art.feature_importance) == 2  # RF and GB only

# Predictions Sanity
assert (preds_df["predicted_RUL"] >= 0).all()  # RUL cannot be negative

print("All checks passed successfully!")
print(f"\nBest Model: {rul_art.best_model_name}")
print(f"Model Used: {rul_art.model_used}")
print("Evaluation Metrics:")
for name, metrics in rul_art.evaluation_metrics.items():
    print(f" {name:30s} RMSE: {metrics['rmse']:.2f} NASA: {metrics['nasa_score']:.1f}")

print(f"\nFeature Importance ({rul_art.best_model_name}):")
if rul_art.best_model_name in rul_art.feature_importance:
    print(rul_art.feature_importance[rul_art.best_model_name])

print("\nSample Predictions (top 5 by risk):")
print(preds_df.sort_values("true_RUL").head(5).to_string(index=False))

[load] Train shape : (20631, 26)
[load] Test shape  : (13096, 26)
[load] RUL entries : 100
All checks passed successfully!

Best Model: gradient_boosting
Model Used: gradient_boosting
Evaluation Metrics:
 linear_regression              RMSE: 19.49 NASA: 608.2
 random_forest                  RMSE: 19.39 NASA: 832.0
 gradient_boosting              RMSE: 18.60 NASA: 698.0

Feature Importance (gradient_boosting):
risk_score        0.681477
health_index      0.203469
HI_velocity       0.111475
HI_variability    0.003578
Name: gradient_boosting_importance, dtype: float64

Sample Predictions (top 5 by risk):
 unit  true_RUL  predicted_RUL  error        model_used
   34         7              4  -3.21 gradient_boosting
   68         8             11   2.96 gradient_boosting
   31         8             13   5.16 gradient_boosting
   81         8              7  -1.49 gradient_boosting
   82         9              6  -2.89 gradient_boosting


In [4]:
import os
import sys
from pathlib import Path

print('cwd:', os.getcwd())
print('sys.path[0]:', sys.path[0])
print('has data/load.py in cwd:', (Path.cwd() / 'data' / 'load.py').exists())
print('has data/load.py in parent:', (Path.cwd().parent / 'data' / 'load.py').exists())

cwd: /Users/arnavhmutt/Desktop/aviation-ds-projects/project-2-predictive-maintenance/notebooks
sys.path[0]: /Users/arnavhmutt/Desktop/aviation-ds-projects/project-2-predictive-maintenance
has data/load.py in cwd: False
has data/load.py in parent: True
